In [ ]:
#!pip install openai

In [37]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

TXT_FOLDER= f"data/txt/"
JSON_STEP1_FOLDER = f"output/json/step_1/"
JSON_STEP2_FOLDER = f"output/json/step_2/"
testi = []
risultati = []

In [38]:
OPENAI_MODEL = "gpt-4o-mini" #gpt-5-mini

In [39]:
SYSTEM_PROMPT = (
    "Sei un estrattore di informazioni. "
    "Devi rispondere SOLO in formato json valido (oggetto JSON), senza testo extra, "
    "senza markdown e senza code fences. "
    "La risposta deve essere un unico oggetto json."    
)

In [40]:
USER_INSTRUCTIONS = """Analizza il testo ed estrai tutte le entità rilevanti (centri, sportelli, case rifugio, associazioni, consorzi, ASL, enti territoriali).
Regole di estrazione e formato:
- Restituisci un UNICO oggetto JSON con una lista sotto la chiave 'entities'.
- Ogni elemento di 'entities' deve contenere i campi:
  - 'nome' (stringa): nome completo come appare nel testo.
  - 'tipo' (stringa): uno tra {
      Centro Antiviolenza, Sportello collegato, Casa Rifugio,
      Associazione di volontariato, Associazione di promozione sociale,
      Consorzio socio-assistenziale, ASL, Ente territoriale, Altro
    }.
  - 'comune' (stringa): valorizza SOLO se esplicitamente presente o ricavabile da normalizzazione geografica.
  - 'indirizzo' (stringa): valorizza SOLO se esplicitamente presente (via/piazza/numero civico).
  - 'ente_capofila' (stringa): valorizza se dal testo emerge esplicitamente chi è l’ente capofila/gestore.
  - 'note' (stringa): dettagli aggiuntivi utili (es. “sportelli decentrati del CAV n.10/A del Cuneese”, “collegamento al 1522”, “nuovo centro”, “in coordinamento con …”).

Regole di copertura (evita perdite di informazione):
- NON limitarti a centri/sportelli/case: includi anche associazioni, consorzi, ASL ed enti territoriali citati come partner o gestori.
- Mantieni SEMPRE 'comune' e 'indirizzo' quando sono presenti nel testo (anche se collegati a un ente partner).
- Se un soggetto è indicato come “ente capofila” o “gestore”, compila 'ente_capofila' con il SUO nome (oltre a riportarlo in 'nome'); usa 'note' per i dettagli extra.
- Se il testo indica “sportelli decentrati” o collegamenti a un CAV esistente, riportalo nelle 'note' dell’ente/centro pertinente.

Normalizzazioni e disambiguazioni:
- Normalizza riferimenti geografici generici in toponimi espliciti quando chiaramente deducibili:
  - “del Cuneese” → comune “Cuneo”.
  - Se è indicata una provincia/area ma non il comune, lascia 'comune' vuoto.
- Conserva e normalizza sigle e abbreviazioni note: ASL (azienda sanitaria), Ass., Ass.ne, Univ., Pref.
- Non inventare dati: se un campo non è presente, lascialo come stringa vuota "".

Deduplicazione e coalescenza:
- Unisci duplicati evidenti dello stesso ente/centro (stesso nome o varianti minime); preferisci la forma più completa.
- Se lo stesso ente ha più ruoli (es. partner e gestore), usa un UNICO record e descrivi i ruoli in 'note'.

Output:
- Rispondi SOLO con un oggetto JSON valido (nessun testo extra, nessun markdown, nessun code fence).
Testo:
"""


In [41]:
def uploadfile(start_reg=6, end_reg=6):    
    for codice in range(start_reg, end_reg + 1):
        # nome della cartella txt
        cartella_txt = f"{TXT_FOLDER}/{codice:02d}"
        # 1️⃣ Carica tutti i file .txt in un array
        testi = []
        file_txt = [f for f in os.listdir(cartella_txt) if f.endswith(".txt")]

        for nome_file in file_txt:
            percorso = os.path.join(cartella_txt, nome_file)
            with open(percorso, "r", encoding="utf-8") as f:
                testo = f.read()
                testi.append({"nome_file": nome_file, "contenuto": testo})

        print(f"📂 Caricati {len(testi)} file di testo.")
        return testi
    
testi = uploadfile(2, 2)
print(f"{testi}")  

📂 Caricati 1 file di testo.
[{'nome_file': '02_2406_rta_02_240904124958_4166---lrn4del25feb2013.txt', 'contenuto': "Consiglio Regionale della Valle d'Aosta - Legge regionale 25 febbraio ...\n\n1di6\n\nLegge regionale 25 febbraio 2013, n. 4 - Testo vigente\nLegge regionale 25 febbraio 2013, n. 4 - Testo vigente\n\nLegge regionale 25 febbraio 2013, n. 4\n\nInterventi di prevenzione e di contrasto alla violenza di genere e misure di sostegno alle\ndonne vittime di violenza di genere.\n\n(B.U. del 12 marzo 2013, n. 11)\n\nArt. 1\n(Oggetto e finalita)\n\n1. La Regione riconosce in ogni forma di violenza di genere una violazione dei diritti\numani, dell'integrita fisica e psicologica, della sicurezza, della liberta e della dignita della\npersona.\n\n2. Ai fini della presente legge, per violenza di genere si intende qualsiasi forma, diretta o\nindiretta, e qualsiasi grado di violenza sessuale, fisica, psicologica ed economica, di\nminaccia di violenza, di molestie, di stalking e di persecuzio

In [42]:
# Chiave API
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [43]:

def extract_entities_with_llm (testi):

    risultati = []
    for item in testi:
        try:
            resp = client.responses.create(
                model=OPENAI_MODEL,
                instructions=SYSTEM_PROMPT,
                input=USER_INSTRUCTIONS + item["contenuto"] + "\n---"
            )
            data = json.loads(resp.output_text)
            risultati.append({
                "file": item["nome_file"],
                "risultato": data
            })
        except Exception as e:
            print(f"❌ Errore con {item['nome_file']}: {e}")
            risultati.append({
                "file": item["nome_file"],
                "risultato": None
            })

    # 3️⃣ Stampa i risultati (primi 2 come esempio)
    print(json.dumps(risultati[:2], indent=4, ensure_ascii=False))
    return risultati
    
risultati = extract_entities_with_llm(testi)

[
    {
        "file": "02_2406_rta_02_240904124958_4166---lrn4del25feb2013.txt",
        "risultato": {
            "entities": [
                {
                    "nome": "Centro donne contro la violenza",
                    "tipo": "Centro Antiviolenza",
                    "comune": "",
                    "indirizzo": "",
                    "ente_capofila": "",
                    "note": "promuove e sostiene l'attività del Centro antiviolenza"
                },
                {
                    "nome": "Azienda USL della Valle d'Aosta",
                    "tipo": "ASL",
                    "comune": "",
                    "indirizzo": "",
                    "ente_capofila": "",
                    "note": "rappresentante nell'osservatorio regionale e nel Forum"
                },
                {
                    "nome": "Consulta per le pari opportunità",
                    "tipo": "Altro",
                    "comune": "",
                    "indirizzo": ""

In [44]:
def print_result(risultati, reg_code):    
    # 📂 Cartella di output
    output_dir = f"{JSON_STEP1_FOLDER}"
    os.makedirs(output_dir, exist_ok=True)

    # 📄 Salva tutti i risultati in un unico file
    output_file = os.path.join(output_dir, f"{reg_code}_risultati.json")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(risultati, f, indent=4, ensure_ascii=False)
        
    print(f"✅ Risultati salvati in: {output_file}")
    
print_result(risultati,"02")

✅ Risultati salvati in: output/json/step_1/02_risultati.json
